# YOLOv8n — LISA Traffic Sign Fine-Tuning
**AAI-590 Capstone Group 2**

Training runs **locally** (CUDA GPU, Apple MPS, or CPU).

| Step | Description |
|------|-------------|
| 0    | Configure paths and auto-detect device |
| 0b   | Verify LISA dataset exists locally |
| 1    | Verify compute device |
| 2    | Convert LISA → YOLO format |
| 3    | Inspect image counts + class distribution |
| 4    | Visualise sample annotated images |
| 5    | Train YOLOv8n |
| 6/7  | Save best.pt → trained_models/ |
| 8    | Run inference on val images |

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  STEP 0 — Local Setup                                                   ║
# ║  Run this first. Everything else depends on variables set here.         ║
# ╚══════════════════════════════════════════════════════════════════════════╝
import sys, shutil, pathlib, torch

# ── Resolve repo root ─────────────────────────────────────────────────────────
# Works whether you launch Jupyter from the repo root or from training/
_cwd = pathlib.Path().resolve()
if _cwd.name == 'training':
    REPO_ROOT = _cwd.parent
elif (_cwd / 'training').is_dir():
    REPO_ROOT = _cwd
else:
    REPO_ROOT = _cwd  # fallback — set manually if needed

TRAINING_DIR = REPO_ROOT / 'training'
sys.path.insert(0, str(TRAINING_DIR))

# ── Paths ─────────────────────────────────────────────────────────────────────
LISA_ROOT  = REPO_ROOT / 'training_data' / 'Lisa'   # local LISA dataset
BASE_MODEL = 'yolov8n.pt'                            # ultralytics auto-downloads
OUT_DIR    = REPO_ROOT / 'trained_models'
YOLO_DATA  = TRAINING_DIR / 'lisa_yolo'
RUN_DIR    = REPO_ROOT / 'runs' / 'train'

# ── Auto-detect device and set batch/workers ──────────────────────────────────
if torch.cuda.is_available():
    props    = torch.cuda.get_device_properties(0)
    vram_gb  = props.total_memory / 1e9
    GPU_NAME = props.name
    DEVICE   = '0'
    if vram_gb >= 35:
        BATCH, WORKERS = 64, 8
    elif vram_gb >= 14:
        BATCH, WORKERS = 32, 4
    else:
        BATCH, WORKERS = 16, 4
elif torch.backends.mps.is_available():   # Apple Silicon
    GPU_NAME = 'Apple MPS'
    vram_gb  = 0
    DEVICE   = 'mps'
    BATCH, WORKERS = 8, 0   # MPS is most stable with workers=0
else:
    GPU_NAME = 'CPU'
    vram_gb  = 0
    DEVICE   = 'cpu'
    BATCH, WORKERS = 4, 2

# ── Training hyper-parameters ─────────────────────────────────────────────────
EPOCHS    = 100
IMG_SIZE  = 640
LR0       = 0.01
PATIENCE  = 20
AUGMENT   = True
FREEZE    = 0
RUN_NAME  = 'lisa_yolov8n'
VAL_SPLIT = 0.15
SEED      = 42

# ── Sanity check ─────────────────────────────────────────────────────────────
vram_str = f'  {vram_gb:.1f} GB VRAM' if vram_gb else ''
print(f'Device     : {DEVICE}  ({GPU_NAME}{vram_str})')
print(f'REPO_ROOT  : {REPO_ROOT}  exists={REPO_ROOT.exists()}')
print(f'LISA_ROOT  : {LISA_ROOT}  exists={LISA_ROOT.exists()}')
print(f'BASE_MODEL : {BASE_MODEL}  (auto-download if not cached)')
print(f'YOLO_DATA  : {YOLO_DATA}')
print(f'DEVICE={DEVICE}  BATCH={BATCH}  WORKERS={WORKERS}  EPOCHS={EPOCHS}')

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  STEP 0b — Verify LISA Dataset Exists Locally                           ║
# ║                                                                          ║
# ║  Expected path: <repo>/training_data/Lisa/                              ║
# ║                                                                          ║
# ║  If missing, download from:                                              ║
# ║    https://cvrr.ucsd.edu/LISA/lisa-traffic-sign-dataset.html            ║
# ║  Then extract so that the folder exists at:                              ║
# ║    <repo>/training_data/Lisa/                                            ║
# ╚══════════════════════════════════════════════════════════════════════════╝
if not LISA_ROOT.exists():
    raise FileNotFoundError(
        f'LISA dataset not found at: {LISA_ROOT}\n\n'
        'Download from: https://cvrr.ucsd.edu/LISA/lisa-traffic-sign-dataset.html\n'
        f'Then extract so the folder exists at:\n  {LISA_ROOT}'
    )

print(f'LISA_ROOT found : {LISA_ROOT}')
subdirs = sorted(p.name for p in LISA_ROOT.iterdir() if p.is_dir())
print(f'Sub-directories ({len(subdirs)}): {subdirs[:8]}{"..." if len(subdirs) > 8 else ""}')

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  STEP 1 — Device Verification                                           ║
# ╚══════════════════════════════════════════════════════════════════════════╝
import torch

print('torch version  :', torch.__version__)
print('CUDA available :', torch.cuda.is_available())
print('MPS  available :', torch.backends.mps.is_available())

if torch.cuda.is_available():
    print(f'GPU name       : {torch.cuda.get_device_name(0)}')
    print(f'VRAM           : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
    print(f'CUDA version   : {torch.version.cuda}')
    x = torch.zeros(1).cuda()
    print(f'Tensor device  : {x.device}')
    print('\nGPU is ready.')
elif torch.backends.mps.is_available():
    x = torch.zeros(1).to('mps')
    print(f'Tensor device  : {x.device}')
    print('\nApple MPS is ready.')
else:
    print('\nNo GPU found — running on CPU (training will be slow).')

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  STEP 2 — Prepare LISA Dataset                                          ║
# ║  Reads LISA_ROOT, writes YOLO format to YOLO_DATA.                      ║
# ║  Skip if YOLO_DATA already exists from a previous run.                  ║
# ╚══════════════════════════════════════════════════════════════════════════╝
import sys
sys.path.insert(0, str(TRAINING_DIR))

if YOLO_DATA.exists():
    print(f'YOLO_DATA already exists at {YOLO_DATA} — skipping conversion.')
else:
    if not LISA_ROOT.exists():
        raise FileNotFoundError(
            f'LISA data not found at {LISA_ROOT}\n'
            'Run Step 0b first.'
        )

    from prepare_lisa import prepare

    yaml_path = prepare(
        lisa_root=str(LISA_ROOT),
        out_root=str(YOLO_DATA),
        val_split=VAL_SPLIT,
        seed=SEED,
    )
    print('\ndataset.yaml:', yaml_path)

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  STEP 3 — Inspect Dataset                                               ║
# ╚══════════════════════════════════════════════════════════════════════════╝
import sys
from collections import Counter
sys.path.insert(0, str(TRAINING_DIR))

from lisa_classes import CLASS_NAMES

train_imgs = list((YOLO_DATA / 'images' / 'train').glob('*.jpg'))
val_imgs   = list((YOLO_DATA / 'images' / 'val').glob('*.jpg'))
print(f'Train images : {len(train_imgs)}')
print(f'Val   images : {len(val_imgs)}')

counter = Counter()
for split in ('train', 'val'):
    for lbl_file in (YOLO_DATA / 'labels' / split).glob('*.txt'):
        for line in lbl_file.read_text().splitlines():
            if line.strip():
                counter[int(line.split()[0])] += 1

print('\nAnnotation counts per class:')
for idx, name in enumerate(CLASS_NAMES):
    print(f'  {idx:2d}  {name:<20s} {counter.get(idx, 0):>6d}')

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  STEP 4 — Visualise Sample Training Images                              ║
# ╚══════════════════════════════════════════════════════════════════════════╝
import random, cv2
import matplotlib.pyplot as plt
import matplotlib.patches as patches

COLORS = [
    (0.2, 0.8, 0.2),  # go
    (0.0, 0.6, 1.0),  # goForward
    (0.0, 1.0, 1.0),  # goLeft
    (1.0, 0.2, 0.2),  # stop
    (1.0, 0.5, 0.0),  # stopLeft
    (1.0, 1.0, 0.0),  # warning
    (0.8, 0.0, 0.8),  # warningLeft
]

random.seed(0)
sample_imgs = random.sample(train_imgs, min(6, len(train_imgs)))

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for ax, img_path in zip(axes.flat, sample_imgs):
    img = cv2.cvtColor(cv2.imread(str(img_path)), cv2.COLOR_BGR2RGB)
    h, w = img.shape[:2]
    ax.imshow(img)
    lbl_path = YOLO_DATA / 'labels' / 'train' / (img_path.stem + '.txt')
    if lbl_path.exists():
        for line in lbl_path.read_text().splitlines():
            parts = line.split()
            if len(parts) < 5:
                continue
            cls_idx = int(parts[0])
            cx, cy, bw, bh = map(float, parts[1:])
            x1 = (cx - bw / 2) * w
            y1 = (cy - bh / 2) * h
            color = COLORS[cls_idx % len(COLORS)]
            ax.add_patch(patches.Rectangle(
                (x1, y1), bw * w, bh * h,
                linewidth=2, edgecolor=color, facecolor='none'
            ))
            ax.text(x1, y1 - 4, CLASS_NAMES[cls_idx],
                    color=color, fontsize=8, fontweight='bold')
    ax.set_title(img_path.name, fontsize=8)
    ax.axis('off')

plt.suptitle('Sample Training Images with LISA Annotations', fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  STEP 5 — Train YOLOv8n                                                 ║
# ╚══════════════════════════════════════════════════════════════════════════╝
from ultralytics import YOLO

yaml_path = YOLO_DATA / 'dataset.yaml'
if not yaml_path.exists():
    raise FileNotFoundError(f'dataset.yaml not found: {yaml_path} — run Step 2 first')

model = YOLO(BASE_MODEL)

train_kwargs = dict(
    data        = str(yaml_path.resolve()),
    epochs      = EPOCHS,
    imgsz       = IMG_SIZE,
    batch       = BATCH,
    workers     = WORKERS,
    device      = DEVICE,
    lr0         = LR0,
    patience    = PATIENCE,
    project     = str(RUN_DIR),
    name        = RUN_NAME,
    exist_ok    = True,
    mosaic      = 1.0 if AUGMENT else 0.0,
    copy_paste  = 0.0,   # disabled — shape mismatch bug on LISA
    freeze      = FREEZE if FREEZE > 0 else None,
    verbose     = True,
    save        = True,
    save_period = -1,
)

print('Training config:')
for k, v in train_kwargs.items():
    print(f'  {k:<12}: {v}')

results = model.train(**train_kwargs)
print('\nTraining complete.')

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  STEP 7 — Save Model to trained_models/                                 ║
# ╚══════════════════════════════════════════════════════════════════════════╝
import shutil

run_weights = RUN_DIR / RUN_NAME / 'weights'
best_pt     = run_weights / 'best.pt'
last_pt     = run_weights / 'last.pt'

OUT_DIR.mkdir(parents=True, exist_ok=True)

dest_best = OUT_DIR / 'yolov8n_lisa_best.pt'
shutil.copy2(best_pt, dest_best)
print('Best model  →', dest_best)

if last_pt.exists():
    dest_last = OUT_DIR / 'yolov8n_lisa_last.pt'
    shutil.copy2(last_pt, dest_last)
    print('Last checkpoint →', dest_last)

print('\nDone. Models saved to', OUT_DIR)

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  STEP 7 — Save Model to Drive                                           ║
# ╚══════════════════════════════════════════════════════════════════════════╝
import shutil

run_weights = RUN_DIR / RUN_NAME / 'weights'
best_pt     = run_weights / 'best.pt'
last_pt     = run_weights / 'last.pt'

OUT_DIR.mkdir(parents=True, exist_ok=True)

dest_best = OUT_DIR / 'yolov8n_lisa_best.pt'
shutil.copy2(best_pt, dest_best)
print('Best model  →', dest_best)

if last_pt.exists():
    dest_last = OUT_DIR / 'yolov8n_lisa_last.pt'
    shutil.copy2(last_pt, dest_last)
    print('Last checkpoint →', dest_last)

print('\nDone. Models saved to Drive.')

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  STEP 8 — Inference on Validation Set                                   ║
# ╚══════════════════════════════════════════════════════════════════════════╝
import random, cv2
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from ultralytics import YOLO

dest_best   = OUT_DIR / 'yolov8n_lisa_best.pt'
infer_model = YOLO(str(dest_best))

COLORS = [
    (0.2, 0.8, 0.2), (0.0, 0.6, 1.0), (0.0, 1.0, 1.0),
    (1.0, 0.2, 0.2), (1.0, 0.5, 0.0), (1.0, 1.0, 0.0), (0.8, 0.0, 0.8),
]

val_imgs_list = list((YOLO_DATA / 'images' / 'val').glob('*.jpg'))
random.seed(1)
sample = random.sample(val_imgs_list, min(6, len(val_imgs_list)))

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for ax, img_path in zip(axes.flat, sample):
    img_bgr = cv2.imread(str(img_path))
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    preds   = infer_model.predict(img_bgr, conf=0.25, verbose=False)[0]
    ax.imshow(img_rgb)
    for box in preds.boxes:
        x1, y1, x2, y2 = box.xyxy[0].tolist()
        cls_idx = int(box.cls[0])
        conf    = float(box.conf[0])
        color   = COLORS[cls_idx % len(COLORS)]
        ax.add_patch(patches.Rectangle(
            (x1, y1), x2 - x1, y2 - y1,
            linewidth=2, edgecolor=color, facecolor='none'
        ))
        ax.text(x1, y1 - 4, f'{CLASS_NAMES[cls_idx]} {conf:.2f}',
                color=color, fontsize=7, fontweight='bold')
    ax.set_title(img_path.name, fontsize=8)
    ax.axis('off')

plt.suptitle('YOLOv8n LISA — Val Set Predictions', fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  OPTIONAL — Resume Training from last checkpoint                        ║
# ╚══════════════════════════════════════════════════════════════════════════╝
from ultralytics import YOLO

last_ckpt = RUN_DIR / RUN_NAME / 'weights' / 'last.pt'
if last_ckpt.exists():
    resume_model = YOLO(str(last_ckpt))
    results = resume_model.train(resume=True)
    print('Resumed training complete.')
else:
    print('No last.pt found at', last_ckpt)